## 1. Dataset Selection and Setup

In [1]:
import pandas as pd
import numpy as np

In [22]:
data = pd.read_csv("data/athletes.csv")

In [16]:
# First we will capture the raw dataset version with DVC.

In [23]:
!dvc add data/athletes.csv


To track the changes with git, run:

	git add 'data\athletes.csv.dvc'

To enable auto staging, run:

	dvc config core.autostage true


\u280b Checking graph



In [24]:
!git add data/.gitignore data/athletes.csv.dvc

In [25]:
!git commit -m "Add raw dataset version"

[detached HEAD d48eee9] Add raw dataset version
 1 file changed, 2 insertions(+), 2 deletions(-)


In [27]:
!git tag dataset-v1

In [28]:
# Then we preprocess the data.

In [32]:
data.head(1)

,athlete_id,name,region,team,affiliate,gender,age,height,weight,fran,...,snatch,deadlift,backsq,pullups,eat,train,background,experience,schedule,howlong
0,2554.0,Pj Ablang,South West,Double Edge,Double Edge CrossFit,Male,24.0,70.0,166.0,NaN,...,NaN,400.0,305.0,NaN,NaN,I workout mostly at a CrossFit Affiliate|I hav...,I played youth or high school level sports|I r...,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 2x a week|,4+ years|


In [33]:
# Remove non-relevant columns
data = data.dropna(
 subset=[
 'region','age','weight','height','howlong',
 'gender','eat','background','experience',
 'schedule','deadlift','candj','snatch','backsq'
 ]
)
data = data.drop(
 columns=[
 'affiliate','team','name','athlete_id',
 'fran','helen','grace','filthy50',
 'fgonebad','run400','run5k','pullups','train'
 ]
)
# Remove outliers
data = data[data['weight'] < 1500]
data = data[data['gender'] != '--']
data = data[data['age'] >= 18]
data = data[(data['height'] < 96) & (data['height'] > 48)]
data = data[
 ((data['gender'] == 'Male') & (data['deadlift'] <= 1105)) |
 ((data['gender'] == 'Female') & (data['deadlift'] <= 636))
]
data = data[(data['candj'] > 0) & (data['candj'] <= 395)]
data = data[(data['snatch'] > 0) & (data['snatch'] <= 496)]
data = data[(data['backsq'] > 0) & (data['backsq'] <= 1069)]
# Clean survey data
decline_dict = {'Decline to answer|': np.nan}
data = data.replace(decline_dict)
data = data.dropna(
 subset=[
 'background','experience',
 'schedule','howlong','eat'
 ]
)

In [34]:
data.head(1)

,region,gender,age,height,weight,candj,snatch,deadlift,backsq,eat,background,experience,schedule,howlong
21,Southern California,Male,30.0,71.0,200.0,235.0,175.0,385.0,315.0,I eat whatever is convenient|,I played youth or high school level sports|I p...,I began CrossFit by trying it alone (without a...,I do multiple workouts in a day 1x a week|I ty...,1-2 years|


In [37]:
data["total_lift"] = (
    data["deadlift"]
    + data["candj"]
    + data["snatch"]
    + data["backsq"]
)

# Below we make sure to handle infinite numbers or nulls, as they can become problematic later on in our training set.
data["total_lift"] = pd.to_numeric(
    data["total_lift"],
    errors="coerce"
)
data["total_lift"] = data["total_lift"].replace(
    [np.inf, -np.inf],
    np.nan
)
data = data.dropna(subset=["total_lift"])

In [39]:
# As part of our feature engineering, we'll loop through and create new columns as our categorized features.
import pandas as pd

columns_to_encode = [
    "eat",
    "background",
    "experience",
    "schedule",
    "howlong",
]

encoded_dfs = []

for column in columns_to_encode:
    encoded = (
        data[column]
        .fillna("")
        .str.get_dummies(sep="|")
        .rename(columns=lambda x: f"{column}_{x.strip()}")
    )

    encoded_dfs.append(encoded)

data = pd.concat(
    [data.drop(columns=columns_to_encode)] + encoded_dfs,
    axis=1,
)

In [43]:
# Confirmed that only 2 genders are provided in the data.
data["gender_male"] = (data["gender"] == "Male").astype(int)
data = data.drop(columns=["gender"])

In [44]:
data.head(1)

,region,age,height,weight,candj,snatch,deadlift,backsq,total_lift,eat_Decline to answer,...,schedule_I typically rest 4 or more days per month,schedule_I typically rest fewer than 4 days per month,schedule_I usually only do 1 workout a day,howlong_1-2 years,howlong_2-4 years,howlong_4+ years,howlong_6-12 months,howlong_Decline to answer,howlong_Less than 6 months,gender_male
21,Southern California,30.0,71.0,200.0,235.0,175.0,385.0,315.0,1110.0,0,...,1,0,0,1,0,0,0,0,0,1


In [46]:
data.to_csv("data/processed.csv")

In [47]:
!dvc add data/processed.csv


To track the changes with git, run:

	git add 'data\.gitignore' 'data\processed.csv.dvc'

To enable auto staging, run:

	dvc config core.autostage true


\u280b Checking graph



In [48]:
!git add data/.gitignore data/processed.csv.dvc

In [49]:
!git commit -m "Add processed dataset version"

[detached HEAD e827681] Add processed dataset version
 2 files changed, 6 insertions(+)
 create mode 100644 Assignment_2/data/processed.csv.dvc


In [52]:
!git tag dataset-v2